In [6]:
import pandas as pd
import numpy as np
import pickle
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# ==========================================
# 1. Load Raw CSV & Filter
# ==========================================
print("Step 1: Loading raw dataset...")
raw_csv_path = 'D:/VishnuMK/Projects/Email/URL_detection/malicious_phish.csv/malicious_phish.csv'

df = pd.read_csv(raw_csv_path)

# Drop nulls and exact URL duplicates
df = df.dropna().drop_duplicates(subset=['url'])

# Filter for Benign and Phishing classes
df = df[df['type'].isin(['benign', 'phishing'])].copy()

# EXPLICIT LABEL MAPPING: Benign MUST be 0, Phishing MUST be 1
df['label'] = df['type'].map({'benign': 0, 'phishing': 1})

# Sanity check label counts
print("\nClass Distribution:")
print(df['type'].value_counts())

urls = df['url'].astype(str).values
labels = df['label'].values

# ==========================================
# 2. Tokenize & Pad Sequences directly
# ==========================================
print("\nStep 2: Tokenizing raw URLs...")
MAX_LEN = 200

tokenizer = Tokenizer(filters='', lower=True, char_level=True, oov_token='<UNK>')
tokenizer.fit_on_texts(urls)

# SAVE THE EXACT TOKENIZER IMMEDIATELY
tokenizer_save_path = 'D:/VishnuMK/Projects/Email/repo/tokenizer.pickle'
with open(tokenizer_save_path, 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)
print(f"Tokenizer saved directly to '{tokenizer_save_path}'")

# Convert URLs to sequences
sequences = tokenizer.texts_to_sequences(urls)
X = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')
y = np.array(labels)

# Set correct vocabulary size (+1 buffer for padding index 0)
vocab_size = len(tokenizer.word_index) + 2

# ==========================================
# 3. Train / Validation / Test Split
# ==========================================
print("\nStep 3: Splitting dataset...")
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Train samples: {X_train.shape[0]} | Validation: {X_val.shape[0]} | Test: {X_test.shape[0]}")

# ==========================================
# 4. Build CNN + LSTM Architecture
# ==========================================
print("\nStep 4: Building model...")
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=32, input_length=MAX_LEN),
    
    Conv1D(filters=64, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    
    LSTM(64),
    
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ==========================================
# 5. Train & Save Model
# ==========================================
model_save_path = 'D:/VishnuMK/Projects/Email/repo/cnn_lstm_phishing_model.keras'

callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ModelCheckpoint(model_save_path, monitor='val_loss', save_best_only=True)
]

print("\nStep 5: Training model...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=256,
    callbacks=callbacks,
    verbose=1
)

# ==========================================
# 6. Evaluation
# ==========================================
print("\nStep 6: Evaluating model on test split...")
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f" Test Accuracy: {test_acc * 100:.2f}%")

y_pred_prob = model.predict(X_test, verbose=0)
y_pred = (y_pred_prob > 0.5).astype(int)

print("\n--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['Benign (0)', 'Phishing (1)']))

# Explicitly save final trained model state
model.save(model_save_path)
print(f"Model successfully saved to '{model_save_path}'")

Step 1: Loading raw dataset...

Class Distribution:
type
benign      428080
phishing     94086
Name: count, dtype: int64

Step 2: Tokenizing raw URLs...
Tokenizer saved directly to 'D:/VishnuMK/Projects/Email/repo/tokenizer.pickle'

Step 3: Splitting dataset...
Train samples: 417732 | Validation: 52217 | Test: 52217

Step 4: Building model...


d:\VishnuMK\Projects\Email\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Step 5: Training model...
Epoch 1/10
1632/1632 ━━━━━━━━━━━━━━━━━━━━ 100s 61ms/step - accuracy: 0.8197 - loss: 0.4739 - val_accuracy: 0.8204 - val_loss: 0.4695
Epoch 2/10
1632/1632 ━━━━━━━━━━━━━━━━━━━━ 460s 282ms/step - accuracy: 0.8947 - loss: 0.2806 - val_accuracy: 0.9515 - val_loss: 0.1385
Epoch 3/10
1632/1632 ━━━━━━━━━━━━━━━━━━━━ 97s 60ms/step - accuracy: 0.9601 - loss: 0.1185 - val_accuracy: 0.9687 - val_loss: 0.0996
Epoch 4/10
1632/1632 ━━━━━━━━━━━━━━━━━━━━ 105s 64ms/step - accuracy: 0.9717 - loss: 0.0896 - val_accuracy: 0.9733 - val_loss: 0.0822
Epoch 5/10
1632/1632 ━━━━━━━━━━━━━━━━━━━━ 100s 61ms/step - accuracy: 0.9757 - loss: 0.0773 - val_accuracy: 0.9766 - val_loss: 0.0723
Epoch 6/10
1632/1632 ━━━━━━━━━━━━━━━━━━━━ 100s 61ms/step - accuracy: 0.9768 - loss: 0.0746 - val_accuracy: 0.9780 - val_loss: 0.0677
Epoch 7/10
1632/1632 ━━━━━━━━━━━━━━━━━━━━ 100s 61ms/step - accuracy: 0.9788 - loss: 0.0670 - val_accuracy: 0.9793 - val_loss: 0.0627
Epoch 8/10
1632/1632 ━━━━━━━━━━━━━━━━━━━━ 